[<img src="https://gitlab.irit.fr/toc/etu-n7/controle-optimal/-/raw/master/ressources/Logo-toulouse-inp-N7.png" alt="N7" height="80"/>](https://gitlab.irit.fr/toc/etu-n7/controle-optimal)
<img src="https://gitlab.irit.fr/toc/ens-n7/texCoursN7/-/raw/main/logo-insa.png" alt="INSA" height="80" style="margin-left:50px"/>

# Méthodes de Runge-Kutta

- Date : 2025-2026
- Durée approximative : 2h45

**Plan.**

1. L'exemple d'étude et l'ordre de convergence
2. Les méthodes d'Euler, de Runge, de Heun et RK4
3. Généralisation : le tableau de Butcher
4. Comparaison des méthodes
5. Limite de précision : erreur d'arrondi
6. Un système instable
7. Contrôle du pas : doublement avec Euler, puis paire emboîtée RK23
8. Comparaison avec un solveur professionnel (Tsit5)

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Introduction
</div>

**Méthode à un pas explicite.** On désire calculer une approximation de la solution, notée $x(\cdot, t_0, x_0)$, sur l'intervalle 
$I := [t_0, t_f]$ du problème de Cauchy
\begin{equation*}
    \dot{x}(t) = f(t, x(t)), \quad x(t_0) = x_0,
\end{equation*}
où $f \colon \mathbb{R} \times \mathbb{R}^n \to \mathbb{R}^n$ est une application suffisamment régulière. 
On considère pour cela une subdivision de l'intervalle $I$ : 
$$
    t_0 < t_1 < \cdots < t_N := t_f
$$

On note $h_i := t_{i+1} - t_i$ pour $i \in \llbracket 0, N-1 \rrbracket$, les **pas** de la subdivision et 
$h_{\max} := \max_i(h_i)$ le pas le plus long. L'idée est de calculer une approximation de 
la solution en les points de discrétisation. On souhaite donc approcher $x(t_i, t_0, x_0)$ pour 
$i \in \llbracket 1, N \rrbracket$. 

On appelle *méthode à un pas explicite*, toute méthode calculant un N-uplet $(x_1, \ldots, x_N) \in {(\mathbb{R}^n)}^N$ tel que la valeur $x_{i+1}$ est calculée en fonction de $t_i$, $h_i$ et $x_i$ de la forme :

$$
    x_{i+1} = x_i + h_i\, \Phi(t_i, x_i, h_i).
$$

Pour approcher la solution, nous utiliserons une méthode à un pas. Si on note $x_h^* := (x_1^*, \ldots, x_N^*)$ une solution au problème discrétisé (obtenue par la méthode à un pas considérée), alors on souhaite que 
$$
    x_i^* \approx x(t_i, t_0, x_0) \quad \text{pour tout } i \in \llbracket 1, N \rrbracket.
$$

**Ordre de convergence.** On considère à partir de maintenant un pas de temps $h$ **uniforme**. Une méthode à un pas explicite est **convergente** si pour toute solution $x(\cdot, t_0, x_0)$, la suite ${(x_i)}_i$ définie par $x_{i+1} = x_i + h\, \Phi(t_i, x_i, h)$ vérifie 
$$
    \max_{1 \le i \le N}\, \|{x(t_i, t_0, x_0) - x_i}\| \to 0 
    \quad\text{quand}\quad h \to 0.
$$

Si la convergence est d'ordre $p$ alors il existe une constante $C \ge 0$ telle que

$$
    E := \max_{1 \le i \le N}\, \|{x(t_i, t_0, x_0) - x_i}\| \le C\, h^p.
$$

Faisons l'hypothèse que $E = K\, h^p$ pour un certain $K \ge 0$. En passant au logarithme, on obtient

$$
    \log(E) = \log(K) + p\, \log(h).
$$

Nous en déduisons que si on trace $\log(E)$ en fonction de $\log(h)$, on doit obtenir une droite de
pente $p$ et d'ordonnée à l'origine $\log(K)$. C'est cette droite que nous allons tracer et ajuster
(par régression) tout au long de ce TP : sa pente donnera $p$, et son ordonnée à l'origine, une fois
repassée à l'échelle originale (par $10^{(\cdot)}$, puisqu'on utilisera $\log_{10}$), donnera $K$ —
c'est cette valeur de $K$ qui sera affichée aux côtés de $p$ à chaque estimation d'ordre.

**Objectif.** Nous allons dans ce TP, implémenter quelques méthodes (à un pas) de [Runge-Kutta](https://fr.wikipedia.org/wiki/Méthodes_de_Runge-Kutta) explicites et 
étudier leur convergence. On appelle *méthode de Runge-Kutta explicite à $s$ étages*, la méthode définie par le schéma

$$
    \begin{aligned}
        k_1 &= f(t_0, x_0) \\
        k_2 &= f(t_0 + c_2 h, x_0 + h a_{21} k_1) \\
        \vdots & \\[-1em]
        k_s &= f(t_0 + c_s h, x_0 + h \sum_{j=1}^{s-1} a_{sj} k_j) \\
        x_1 &= x_0 + h \sum_{i=1}^{s} b_i k_i,
    \end{aligned}
$$
où les coefficients $c_i$, $a_{ij}$ et $b_i$ sont des constantes qui définissent précisément le schéma. 
On supposera toujours dans la suite que $c_1 = 0$ et $c_i = \sum_{j=1}^{i-1} a_{ij}$, pour $i = 2, \ldots, s$.

In [ ]:
# Activation de l'environnement Julia associé à ce notebook
using Pkg
Pkg.activate(@__DIR__)

# Chargement des paquets
using LinearAlgebra
using OrdinaryDiffEqTsit5
using Plots
using Polynomials

# Figures vectorielles, nettes à tout niveau de zoom, avec des marges
# suffisantes pour ne pas tronquer les labels des axes
default(fmt = :svg, left_margin = 6Plots.mm, bottom_margin = 6Plots.mm)

In [ ]:
# Fonctions auxiliaires

"""
    method_infos(method::Symbol)

Renvoie le triplet `(f_methode, nom, s)` associé au symbole `method`
(`:euler`, `:runge`, `:heun`, `:rk4` ou `:rk38`) : la fonction d'intégration
elle-même, un nom lisible pour les légendes, et le nombre d'étages `s` de la
méthode (utile pour compter le nombre d'évaluations de `f`).
"""
function method_infos(method)
    if method == :euler
        return euler, "Euler", 1
    elseif method == :runge
        return runge, "Runge", 2
    elseif method == :heun
        return heun, "Heun", 3
    elseif method == :rk4
        return rk4, "RK4", 4
    elseif method == :rk38
        return rk38, "RK4 (3/8)", 4
    else
        error("Méthode d'intégration inconnue : $method")
    end
end

"""
    convergence(method, f, x0, tspan, sol, Nspan)

Pour chaque valeur de `N` dans `Nspan`, intègre `f` par la méthode `method`
sur `tspan` avec `N` pas et superpose la trajectoire obtenue à la solution
exacte `sol`. Renvoie un vecteur de graphiques, un par valeur de `N`.
"""
function convergence(method, f, x0, tspan, sol, Nspan)
    method_func, method_name, _ = method_infos(method)
    println("Méthode d'intégration : ", method_name)

    plots = Any[]
    for N in Nspan
        ts, xs = method_func(f, x0, tspan, N)
        h = (tspan[2] - tspan[1]) / N
        plt = plot(
            ts,
            xs;
            label = method_name,
            marker = :circle,
            xlabel = "t",
            ylabel = "x(t)",
            title = "h = $(round(h, digits=2))",
            xlims = (tspan[1] - 0.1, tspan[2] + 0.1),
        )
        plot!(plt, sol; label = "Solution exacte")
        push!(plots, plt)
    end

    return plots
end

const hspan_default = 10 .^ range(-4, stop = 0, length = 20)
const nfe_span_default = 10 .^ range(1, stop = 4, length = 20)

const _SUP_CHARS = Dict('0' => '⁰', '1' => '¹', '2' => '²', '3' => '³', '4' => '⁴',
                         '5' => '⁵', '6' => '⁶', '7' => '⁷', '8' => '⁸', '9' => '⁹',
                         '-' => '⁻')

exposant_str(n::Integer) = join(_SUP_CHARS[c] for c in string(n))

"""
Ticks « jolis » pour un axe logarithmique : une puissance de 10 sur deux
(ou toutes les puissances si l'étendue est petite), avec un label du type
10⁻⁴, 10⁰, 10⁴, ...
"""
function ticks_log10(lo, hi)
    e_lo = floor(Int, log10(lo))
    e_hi = ceil(Int, log10(hi))
    step = (e_hi - e_lo) > 8 ? 2 : 1
    e_start = step * cld(e_lo, step)
    exposants = e_start:step:e_hi
    positions = 10.0 .^ collect(exposants)
    labels = ["10" * exposant_str(e) for e in exposants]
    return (positions, labels)
end

"""
Calcule, pour chaque point (h, erreur), s'il est conservé pour l'estimation de
l'ordre : on cherche le plus long segment, en h décroissant, sur lequel
l'erreur décroît strictement (le régime asymptotique), ce qui écarte à la
fois d'éventuels pas trop grands (régime pré-asymptotique) et les pas trop
petits (régime dominé par l'erreur d'arrondi, cf. § Limite de précision :
erreur d'arrondi). Les points écartés sont affichés en creux (cercle vide)
plutôt qu'en plein.
"""
function masque_avant_plancher(h, errors)
    perm = sortperm(h; rev = true)   # du plus grand pas au plus petit
    n = length(perm)
    best_start, best_len = 1, 1
    cur_start, cur_len = 1, 1
    for k in 2:n
        if errors[perm[k]] < errors[perm[k-1]]
            cur_len += 1
        else
            cur_start, cur_len = k, 1
        end
        if cur_len > best_len
            best_start, best_len = cur_start, cur_len
        end
    end
    garde = falses(length(h))
    garde[perm[best_start:(best_start + best_len - 1)]] .= true
    return garde
end

"""
    ordre(method, f, x0, tspan, sol, hspan, nfespan; xlims, ylims, xlims_nfe, ylims_nfe)

Estime l'ordre de convergence de `method` par régression log-log de l'erreur
globale en fonction du pas `h` (parcourant `hspan`), puis en fonction du
nombre d'évaluations de `f` (parcourant `nfespan`). Renvoie les deux
graphiques correspondants. Délègue le calcul à `ordre!`.
"""
function ordre(method, f, x0, tspan, sol, hspan, nfespan;
               xlims = (1e-4, 1e0), ylims = (1e-14, 1e1),
               xlims_nfe = (1e1, 1e4), ylims_nfe = (1e-14, 1e1))
    plots = [plot(xaxis = :log, yaxis = :log) for _ in 1:2]
    ordre!(plots, method, f, x0, tspan, sol, hspan, nfespan;
           xlims, ylims, xlims_nfe, ylims_nfe)

    plot!(plots[1]; xlabel = "Pas h", ylabel = "Erreur globale E", legend = :topleft)
    plot!(plots[2]; xlabel = "Nombre d'évaluations de f", ylabel = "Erreur globale E", legend = :topright)
    return plots
end

"""
    ordre!(plots, method, f, x0, tspan, sol, hspan, nfespan; xlims, ylims, xlims_nfe, ylims_nfe)

Variante mutante de `ordre` : ajoute la courbe de `method` aux deux
graphiques `plots` déjà existants (utile pour superposer plusieurs méthodes,
cf. § Comparaison des méthodes), en écartant automatiquement les points
contaminés par l'erreur d'arrondi ou pré-asymptotiques via
`masque_avant_plancher` avant la régression.
"""
function ordre!(plots, method, f, x0, tspan, sol, hspan, nfespan;
                xlims, ylims, xlims_nfe, ylims_nfe)
    method_func, method_name, stages = method_infos(method)
    duration = tspan[2] - tspan[1]

    # Couleur de la série, cohérente entre les deux sous-graphiques.
    pal = palette(:default)
    c = pal[mod1(length(plots[1].series_list) + 1, length(pal))]

    # Le nombre de pas est entier : on utilise le pas réellement calculé.
    Nspan = max.(1, round.(Int, duration ./ hspan))
    h_actual = duration ./ Nspan
    errors = [begin
        ts, xs = method_func(f, x0, tspan, N)
        maximum(abs.(xs .- sol.(ts)))
    end for N in Nspan]

    garde = masque_avant_plancher(h_actual, errors)

    regression = fit(log10.(h_actual[garde]), log10.(errors[garde]), 1)
    order = regression[1]
    constant = 10 ^ regression[0]
    println("\nMéthode d'intégration : ", method_name)
    println("ordre de convergence : p = $(round(order, digits = 5))")
    println("constante K (telle que E ≈ K hᵖ) : K = $(round(constant, digits = 5))")

    plot!(plots[1], h_actual, errors; label = method_name, linecolor = c,
          markershape = :circle, markerstrokecolor = c,
          markercolor = ifelse.(garde, c, :white), markerstrokewidth = 1.5,
          xlims, ylims, xticks = ticks_log10(xlims...), yticks = ticks_log10(ylims...))

    # Pour comparer le coût, on affiche le nombre d'appels effectivement réalisés.
    Nspan_nfe = max.(1, round.(Int, nfespan ./ stages))
    nfe_actual = stages .* Nspan_nfe
    errors_nfe = [begin
        ts, xs = method_func(f, x0, tspan, N)
        maximum(abs.(xs .- sol.(ts)))
    end for N in Nspan_nfe]
    plot!(plots[2], nfe_actual, errors_nfe;
          label = method_name, color = c, marker = :circle, xlims = xlims_nfe, ylims = ylims_nfe,
          xticks = ticks_log10(xlims_nfe...), yticks = ticks_log10(ylims_nfe...))

    return plots
end

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
L'exemple d'étude
</div>

On s'intéresse dans ce TP au problème de Cauchy

$$
    \dot{x}(t) = (1-2t) x(t), \quad x(0) = x_0 = 1
$$

sur l'intervalle $[0, 3]$.

In [ ]:
# Définition du problème de Cauchy
f(t, x) = (1-2t)x    # Second membre f(t, x)
x0 = 1.0             # Condition initiale
tspan = (0.0, 3.0);  # Intervalle de temps

### Exercice 1

* Calculer la solution analytique de l'exemple ci-dessus et la coder dans la fonction `sol` ci-dessous.

In [ ]:
# Solution analytique
"""
    sol(t)

Solution analytique du problème de Cauchy de l'exemple d'étude,
x'(t) = (1 - 2t) x(t), x(0) = 1, servant de référence exacte pour
mesurer l'erreur globale des méthodes numériques.
"""
function sol(t)
    return 0 # à compléter
end

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
La méthode d'Euler
</div>

On rappelle que la méthode d'Euler est donnée par :

$$
\left\{
\begin{array}{l}
x_{n+1} = x_n + h f(t_n, x_n) \\
x_0 = x(t_0)
\end{array}
\right.
$$

### Exercice 2

1. Implémenter la méthode d'Euler dans la fonction `euler` ci-dessous.
2. Vérifier la convergence de la méthode d'Euler sur l'exemple donné dans la cellule d'après.

In [ ]:
# Implémentation de la méthode d'Euler
"""
    euler(f, x0, tspan, N)

Méthode d'Euler explicite (ordre 1) : intègre x'(t) = f(t, x(t)),
x(t0) = x0, sur `tspan = (t0, tf)` avec `N` pas de taille constante
`h = (tf - t0) / N`. Renvoie les instants `ts` et les valeurs `xs` calculées
(vecteurs de longueur `N + 1`, incluant la condition initiale).
"""
function euler(f, x0, tspan, N)
    # à compléter
end

In [ ]:
# Convergence de la méthode sur l'exemple
method = :euler
Nspan  = [5, 10, 20]

# Calcul des graphiques
plts = convergence(method, f, x0, tspan, sol, Nspan)

# Affichage des graphiques
plot(plts..., layout=(1, length(Nspan)), size=(800, 400))

3. Afficher l'erreur globale de la méthode d'Euler en fonction de $h$ et vérifier que l'erreur est bien en $O(h)$.

In [ ]:
# Ordre de convergence de la méthode
test_nfe_span = nfe_span_default[10 .≤ nfe_span_default .≤ 1e4]

# Calcul et affichage des erreurs
plots_order_euler = ordre(:euler, f, x0, tspan, sol, hspan_default, test_nfe_span)
plot(plots_order_euler..., layout = (1, 2), size = (800, 400))

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
La méthode de Runge 
</div>

On rappelle que la méthode de Runge est donnée par :

$$
\left\{
\begin{array}{l}
\displaystyle x_{n+1} = x_n + h\, f\left(t_{n} + \frac{h}{2}, ~x_n + \frac{h}{2} f(t_n, x_n)\right) \\
x_0 = x(t_0)
\end{array}
\right.
$$

### Exercice 3

1. Implémenter la méthode de Runge dans la fonction `runge` ci-dessous.
2. Vérifier la convergence de la méthode de Runge sur l'exemple donné dans la cellule d'après.
3. Calculer l'erreur globale de la méthode de Runge en fonction de $h$ et vérifier que l'erreur est bien en $O(h^2)$.

In [ ]:
# Implémentation de la méthode de Runge
"""
    runge(f, x0, tspan, N)

Méthode de Runge, ou méthode du point milieu (ordre 2, 2 étages) : mêmes
conventions d'arguments et de retour que `euler`, avec la mise à jour
x_{i+1} = x_i + h f(t_i + h/2, x_i + (h/2) f(t_i, x_i)).
"""
function runge(f, x0, tspan, N)
    # à compléter
end

In [ ]:
# Convergence de la méthode sur l'exemple
method = :runge
Nspan  = [5, 10, 20]

# Calcul des graphiques
plts = convergence(method, f, x0, tspan, sol, Nspan)

# Affichage des graphiques
plot(plts..., layout=(1, length(Nspan)), size=(800, 400))

In [ ]:
# Ordre de convergence de la méthode
test_nfe_span = nfe_span_default[10 .≤ nfe_span_default .≤ 1e4]

# Calcul et affichage des erreurs
plots_order_runge = ordre(:runge, f, x0, tspan, sol, hspan_default, test_nfe_span)
plot(plots_order_runge..., layout = (1, 2), size = (800, 400))

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
La méthode de Heun 
</div>

On rappelle que la méthode de Heun est donnée par :

$$
\left\{
\begin{array}{l}
    \displaystyle k_1 = f(t_n, x_n) \\[1em]
    \displaystyle k_2 = f(t_n + \frac{h}{3}, x_n + \frac{h}{3} k_1) \\[1em]
    \displaystyle k_3 = f(t_n + \frac{2}{3}h, x_n + \frac{2}{3}h\, k_2) \\[1em]
    \displaystyle x_{n+1} = x_n + h\left(\frac{1}{4} k_1 + \frac{3}{4} k_3\right), \quad 
    \displaystyle x_0 = x(t_0)
\end{array}
\right.
$$

### Exercice 4

1. Implémenter la méthode de Heun dans la fonction `heun` ci-dessous.
2. Vérifier la convergence de la méthode de Heun sur l'exemple donné dans la cellule d'après.
3. Calculer l'erreur globale de la méthode de Heun en fonction de $h$ et vérifier que l'erreur est bien en $O(h^3)$.

In [ ]:
# Implémentation de la méthode de Heun
"""
    heun(f, x0, tspan, N)

Méthode de Heun à 3 étages (ordre 3) : mêmes conventions que `euler` et
`runge`, avec le tableau de Butcher c = (0, 1/3, 2/3), b = (1/4, 0, 3/4).
"""
function heun(f, x0, tspan, N)
    # à compléter
end

In [ ]:
# Convergence de la méthode sur l'exemple
method = :heun
Nspan  = [5, 10, 20]

# Calcul des graphiques
plts = convergence(method, f, x0, tspan, sol, Nspan)

# Affichage des graphiques
plot(plts..., layout=(1, length(Nspan)), size=(800, 400))

In [ ]:
# Ordre de convergence de la méthode
test_nfe_span = nfe_span_default[10 .≤ nfe_span_default .≤ 1e4]

# Calcul et affichage des erreurs
plots_order_heun = ordre(:heun, f, x0, tspan, sol, hspan_default, test_nfe_span)
plot(plots_order_heun..., layout = (1, 2), size = (800, 400))

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
La méthode de Runge-Kutta d'ordre 4
</div>

On rappelle que la méthode de Runge-Kutta d'ordre 4 est donnée par :

$$
\left\{
\begin{array}{l}
    \displaystyle k_1 = f(t_n, x_n) \\[1em]
    \displaystyle k_2 = f(t_n + \frac{h}{2}, x_n + \frac{h}{2} k_1) \\[1em]
    \displaystyle k_3 = f(t_n + \frac{h}{2}, x_n + \frac{h}{2} k_2) \\[1em]
    \displaystyle k_4 = f(t_n + h, x_n + h\, k_3) \\[1em]
    \displaystyle x_{n+1} = x_n + \frac{h}{6} (k_1 + 2 k_2 + 2 k_3 + k_4), \quad 
    \displaystyle x_0 = x(t_0)
\end{array}
\right.
$$

### Exercice 5

1. Implémenter la méthode de Runge-Kutta d'ordre 4 dans la fonction `rk4` ci-dessous.
2. Vérifier la convergence de la méthode de Runge-Kutta d'ordre 4 sur l'exemple donné dans la cellule d'après.
3. Calculer l'erreur globale de la méthode de Runge-Kutta d'ordre 4 en fonction de $h$ et vérifier que l'erreur est bien en $O(h^4)$.

In [ ]:
# Implémentation de la méthode de Runge-Kutta d'ordre 4
"""
    rk4(f, x0, tspan, N)

Méthode de Runge-Kutta classique à 4 étages (ordre 4) : mêmes conventions
que `euler`, `runge` et `heun`, avec le tableau de Butcher usuel
c = (0, 1/2, 1/2, 1), b = (1/6, 1/3, 1/3, 1/6).
"""
function rk4(f, x0, tspan, N)
    # à compléter
end

In [ ]:
# Convergence de la méthode sur l'exemple
method = :rk4
Nspan  = [5, 10, 20]

# Calcul des graphiques
plts = convergence(method, f, x0, tspan, sol, Nspan)

# Affichage des graphiques
plot(plts..., layout=(1, length(Nspan)), size=(800, 400))

In [ ]:
# Ordre de convergence de la méthode
test_nfe_span = nfe_span_default[10 .≤ nfe_span_default .≤ 1e4]

# Calcul et affichage des erreurs
plots_order_rk4 = ordre(:rk4, f, x0, tspan, sol, hspan_default, test_nfe_span)
plot(plots_order_rk4..., layout = (1, 2), size = (800, 400))

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Généralisation : le tableau de Butcher
</div>

Les quatre méthodes précédentes sont des cas particuliers du schéma général rappelé en
introduction. On représente en pratique ce schéma par le **tableau de Butcher**

$$
    \begin{array}{c | c c c c c}
        c_1     & & & & & \\
        c_2     & a_{21} & & & & \\
        c_3     & a_{31} & a_{32} & & & \\
        \vdots  & \vdots & \vdots & \ddots & & \\
        c_s     & a_{s1} & a_{s2} & \cdots & a_{s,s-1} & \\
        \hline
                & b_1 & b_2 & \cdots & b_{s-1} & b_s \\
    \end{array}
$$

c'est-à-dire par la matrice (strictement inférieure) $A = (a_{ij})$ et les vecteurs $b$ et $c$.
Plutôt que de recoder chaque méthode à la main, on peut écrire **une seule fonction**
générique prenant ce tableau en argument.

### Exercice 6

1. Compléter la fonction `rk` ci-dessous, qui implémente le schéma de Runge-Kutta explicite
   à $s$ étages associé à un tableau de Butcher $(A, b, c)$ quelconque.
2. Vérifier, en exécutant la cellule d'après, qu'en lui donnant les tableaux d'Euler, de
   Runge, de Heun et de RK4, on retrouve bien les trajectoires calculées par les fonctions
   dédiées.
3. Instancier le tableau de la règle 3/8 (RK4, autre variante à 4 étages, également d'ordre 4)
   $$
   \begin{array}{c | c c c c}
        0       & & & & \\
        1/3     & 1/3 & & & \\
        2/3     & -1/3 & 1 & & \\
        1       & 1 & -1 & 1 & \\
        \hline
                & 1/8 & 3/8 & 3/8 & 1/8 \\
    \end{array}
   $$
   et vérifier numériquement, avec la fonction `rk`, que son ordre de convergence est bien 4
   (le graphique n'utilise, pour la régression, que les points affichés en disque plein —
   les points en creux sont exclus car dans le régime dominé par l'erreur d'arrondi,
   cf. § Limite de précision : erreur d'arrondi).

In [ ]:
# Implémentation générique à partir d'un tableau de Butcher (A, b, c)
"""
    rk(f, x0, tspan, N, A, b, c)

Méthode de Runge-Kutta explicite générique à `s = length(c)` étages, définie
par le tableau de Butcher `(A, b, c)` (`A` strictement triangulaire
inférieure de taille `s × s`, `b` et `c` de longueur `s`) :
```
k_i = f(t_i + c_i h,  x_i + h ∑_{j<i} A_{ij} k_j),   i = 1, …, s
x_{i+1} = x_i + h ∑_{i=1}^s b_i k_i
```
Retrouve `euler`, `runge`, `heun` et `rk4` pour les tableaux adéquats
(§ Tableaux de Butcher). Mêmes conventions de retour que `euler`.
"""
function rk(f, x0, tspan, N, A, b, c)
    # à compléter
end

In [ ]:
# Tableaux de Butcher des quatre méthodes précédentes
A_euler = zeros(1, 1);                               b_euler = [1.0];              c_euler = [0.0]
A_runge = [0.0 0.0; 1/2 0.0];                         b_runge = [0.0, 1.0];         c_runge = [0.0, 1/2]
A_heun  = [0.0 0.0 0.0; 1/3 0.0 0.0; 0.0 2/3 0.0];    b_heun  = [1/4, 0.0, 3/4];    c_heun  = [0.0, 1/3, 2/3]
A_rk4   = [0.0 0.0 0.0 0.0; 1/2 0.0 0.0 0.0; 0.0 1/2 0.0 0.0; 0.0 0.0 1.0 0.0]
b_rk4   = [1/6, 1/3, 1/3, 1/6];                       c_rk4   = [0.0, 1/2, 1/2, 1.0]

# On compare, pour un même nombre de pas, la trajectoire donnée par `rk`
# à celle donnée par la fonction dédiée
for (name, A, b, c, method_func) in [( "Euler", A_euler, b_euler, c_euler, euler),
                                      ("Runge", A_runge, b_runge, c_runge, runge),
                                      ("Heun ",  A_heun,  b_heun,  c_heun,  heun),
                                      ("RK4  ",   A_rk4,   b_rk4,   c_rk4,   rk4)]
    _, xs_generic = rk(f, x0, tspan, 20, A, b, c)
    _, xs_ref     = method_func(f, x0, tspan, 20)
    println(name, " : écart maximal avec la fonction dédiée = ", maximum(abs.(xs_generic .- xs_ref)))
end

In [ ]:
# Tableau de la règle 3/8 (RK4, ordre 4)
A_38 = [0.0 0.0 0.0 0.0; 1/3 0.0 0.0 0.0; -1/3 1.0 0.0 0.0; 1.0 -1.0 1.0 0.0]
b_38 = [1/8, 3/8, 3/8, 1/8]
c_38 = [0.0, 1/3, 2/3, 1.0]

rk38(f, x0, tspan, N) = rk(f, x0, tspan, N, A_38, b_38, c_38)

# Ordre de convergence de la règle 3/8, par régression log-log
Nspan_38 = max.(1, round.(Int, (tspan[2] - tspan[1]) ./ hspan_default))
h_actual_38 = (tspan[2] - tspan[1]) ./ Nspan_38
errors_38 = [begin
    ts, xs = rk38(f, x0, tspan, N)
    maximum(abs.(xs .- sol.(ts)))
end for N in Nspan_38]

garde_38 = masque_avant_plancher(h_actual_38, errors_38)

regression_38 = fit(log10.(h_actual_38[garde_38]), log10.(errors_38[garde_38]), 1)
println("ordre de convergence estimé pour la règle 3/8 : ", round(regression_38[1], digits = 3))

c_38_couleur = palette(:default)[1]
plot(h_actual_38, errors_38; xaxis = :log, yaxis = :log, linecolor = c_38_couleur,
     markershape = :circle, markerstrokecolor = c_38_couleur,
     markercolor = ifelse.(garde_38, c_38_couleur, :white), markerstrokewidth = 1.5,
     xticks = ticks_log10(extrema(h_actual_38)...), yticks = ticks_log10(extrema(errors_38)...),
     ylims = (1e-14, 10),
     xlabel = "Pas h", ylabel = "Erreur globale E", label = "RK4 (règle 3/8)", legend = :topleft)

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Comparaison des méthodes
</div>

La fonction `ordre!` (§ Fonctions auxiliaires) permet de superposer plusieurs méthodes sur
les mêmes axes. Exécuter la cellule ci-dessous, qui superpose Euler, Runge, Heun, RK4 et RK4 (règle 3/8) :
à gauche l'erreur globale en fonction du pas $h$, à droite en fonction du nombre
d'évaluations de $f$ (le coût réel d'un calcul, puisque chaque étage coûte un appel à $f$).
Sur le graphique de gauche, les points affichés en creux (cercle vide) sont exclus de la
régression : au-delà d'un certain pas, propre à chaque méthode, l'erreur cesse de décroître
et se met même à remonter — un phénomène étudié en détail dans la section suivante
(§ Limite de précision : erreur d'arrondi). On descend ici jusqu'à $h = 10^{-6}$, assez
profond pour voir RK4 et Heun clairement repartir à la hausse, et Runge amorcer sa remontée ;
Euler, en revanche, ne montre rien de tel sur cette plage : sa méthode étant d'ordre $1$, son
propre plancher se situe bien plus bas (il faudrait un nombre de pas $N$ de l'ordre du
milliard pour l'atteindre, hors de portée ici).

### Exercice 7

1. Pour une précision cible de $10^{-8}$, quelle méthode choisir si l'on raisonne en
   nombre de pas ? Ce choix change-t-il si l'on raisonne en nombre d'évaluations de $f$ ?
2. Une méthode d'ordre plus élevé est-elle toujours la plus intéressante, y compris pour
   une précision cible modeste (par exemple $10^{-2}$) ?
3. Repérer, pour RK4, Heun et Runge, le pas $h$ auquel l'erreur cesse de décroître.
   Cet ordre d'apparition (RK4 en premier, puis Heun, puis Runge) est-il cohérent avec la
   formule $h^\star = O(\mathrm{eps}_{\mathrm{mach}}^{1/(p+1)})$ que nous établirons dans
   l'Exercice 8 (section suivante) ?

In [ ]:
# Superposition des cinq méthodes
hspan_compare_deep = 10 .^ range(-6, stop = 0, length = 30)
test_nfe_span_compare = nfe_span_default[10 .≤ nfe_span_default .≤ 1e4]

plots_compare = [plot(xaxis = :log, yaxis = :log) for _ in 1:2]
for method in (:euler, :runge, :heun, :rk4, :rk38)
    ordre!(plots_compare, method, f, x0, tspan, sol, hspan_compare_deep, test_nfe_span_compare;
           xlims = (1e-6, 1e0), ylims = (1e-15, 1e1),
           xlims_nfe = (1e1, 1e4), ylims_nfe = (1e-13, 1e1))
end
plot!(plots_compare[1]; xlabel = "Pas h", ylabel = "Erreur globale E", legend = :bottomright)
plot!(plots_compare[2]; xlabel = "Nombre d'évaluations de f", ylabel = "Erreur globale E", legend = :topright)

plot(plots_compare..., layout = (1, 2), size = (800, 400))

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Limite de précision : erreur d'arrondi
</div>

Les calculs se font en flottant double précision : on note
$\mathrm{eps}_{\mathrm{mach}} \simeq 2.22 \times 10^{-16}$ (`eps(Float64)` en Julia) le plus
petit nombre tel que $1 + \mathrm{eps}_{\mathrm{mach}}$ soit représentable et différent
de $1$. Jusqu'ici, tous les tracés d'ordre de convergence se sont arrêtés bien avant que
cette limite ne se fasse sentir. Que se passe-t-il si l'on prend $h$ beaucoup plus petit ?

### Exercice 8

1. Reprendre le tracé de l'erreur globale de RK4 en fonction de $h$ (cellule d'après), mais
   en étendant `hspan` jusqu'à $h \sim 10^{-6}$.
2. Repérer approximativement le pas $h^\star$ qui minimise l'erreur globale. Que se passe-t-il
   pour $h < h^\star$ ?
3. L'erreur de troncature globale décroît en $O(h^4)$, tandis que l'erreur d'arrondi accumulée
   sur $N = (t_f-t_0)/h$ pas croît grossièrement en $O(\mathrm{eps}_{\mathrm{mach}}/h)$.
   En minimisant la somme de ces deux termes, montrer que $h^\star = O(\mathrm{eps}_{\mathrm{mach}}^{1/5})$
   et comparer à la valeur observée à la question précédente.

In [ ]:
# Erreur globale de RK4, avec un pas descendant jusqu'à l'échelle de l'erreur d'arrondi
hspan_deep = 10 .^ range(-6, stop = 0, length = 30)
Nspan_deep = max.(1, round.(Int, (tspan[2] - tspan[1]) ./ hspan_deep))
h_deep = (tspan[2] - tspan[1]) ./ Nspan_deep
errors_deep = [begin
    ts, xs = rk4(f, x0, tspan, N)
    maximum(abs.(xs .- sol.(ts)))
end for N in Nspan_deep]

# Valeur théorique h* = O(eps_mach^(1/(p+1))) (Exercice 8, question 3), à
# comparer au minimum observé numériquement sur le graphique ci-dessous.
p_rk4 = 4
h_star_theorique = eps(Float64)^(1 / (p_rk4 + 1))
h_star_numerique = h_deep[argmin(errors_deep)]
println("h* théorique (ordre de grandeur) : ", h_star_theorique)
println("h* observé numériquement          : ", h_star_numerique)

plot(h_deep, errors_deep; xaxis = :log, yaxis = :log, marker = :circle,
     xticks = ticks_log10(extrema(h_deep)...), yticks = ticks_log10(extrema(errors_deep)...),
     ylims = (1e-14, 10),
     xlabel = "Pas h", ylabel = "Erreur globale E", label = "RK4", legend = :topleft)
vline!([h_star_theorique]; label = "h* théorique ~ eps_mach^(1/5)", linestyle = :dash, color = :black)

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Un système instable
</div>

Toutes les fonctions écrites jusqu'ici (`euler`, `runge`, `heun`, `rk4`, `rk`) sont génériques :
rien n'y suppose que $x$ soit un scalaire. On les applique maintenant, sans les modifier, au
problème à valeur initiale

$$
    \left\{\begin{array}{l}
        \dot{x}_1(t) = x_1(t) + x_2(t) + \sin(t) \\
        \dot{x}_2(t) = -x_1(t) + 3\, x_2(t) \\
        x_1(0) = -9/25, \quad x_2(0) = -4/25,
    \end{array}\right.
$$

sur $[0, 10]$, dont on vérifie que la solution est

$$
    x_1(t) = -\frac{1}{25}(13 \sin t + 9 \cos t), \qquad
    x_2(t) = -\frac{1}{25}(3 \sin t + 4 \cos t).
$$

### Exercice 9

1. Coder le second membre `f2(t, x)` (à valeurs dans $\mathbb{R}^2$) et la solution exacte
   `sol2(t)` ci-dessous.
2. Exécuter la cellule d'après, qui intègre le système par RK4 à pas fixe pour différentes
   valeurs de $N$, affiche l'erreur globale (au sens de la norme euclidienne) et trace le
   graphique log-log correspondant (comme aux exercices précédents), pour lire l'ordre de
   convergence et la constante $K$. Commentaires : que se passe-t-il pour $N$ trop petit ?
3. Le second membre linéaire s'écrit $\dot{x} = Bx + g(t)$ avec
   $B = \begin{pmatrix} 1 & 1 \\ -1 & 3 \end{pmatrix}$, dont $2$ est valeur propre double.
   Sur $[t_0, t_f] = [0, 10]$, à quoi correspond, dans la Proposition de stabilité du cours
   (constante $S = e^{L(t_f - t_0)}$, notée $L$ pour ne pas la confondre avec la constante
   $K$ de l'ordre de convergence), un facteur d'amplification de l'ordre de
   $e^{2 \times 10} \approx 5 \times 10^8$ ? En quoi cela explique-t-il le comportement observé
   à la question précédente, alors que la solution exacte, elle, reste parfaitement bornée ?


In [ ]:
# Second membre et solution exacte du système
"""
    f2(t, x)

Second membre du système linéaire instable de l'Exercice 9 :
x1'(t) = x1 + x2 + sin(t), x2'(t) = -x1 + 3 x2.
"""
function f2(t, x)
    return [0.0, 0.0] # à compléter
end

"""
    sol2(t)

Solution analytique du système de `f2`, pour la condition initiale
x(0) = (-9/25, -4/25), servant de référence pour mesurer l'erreur globale.
"""
function sol2(t)
    return [0.0, 0.0] # à compléter
end

x0_2 = [-9/25, -4/25]
tspan_2 = (0.0, 10.0)

In [ ]:
# Intégration par RK4 à pas fixe, pour différentes valeurs de N
Nspan_2 = [100, 200, 400, 800, 1600, 3200, 6400]
h_2 = (tspan_2[2] - tspan_2[1]) ./ Nspan_2
errors_2 = [begin
    ts, xs = rk4(f2, x0_2, tspan_2, N)
    maximum(norm.(xs .- sol2.(ts)))
end for N in Nspan_2]

for (N, err) in zip(Nspan_2, errors_2)
    println("N = $N, \t erreur globale = $err")
end

# Même chose sous forme de graphique log-log, pour lire l'ordre de convergence et la
# constante K (comme aux exercices précédents, avec RK4 sur l'exemple d'étude)
garde_2 = masque_avant_plancher(h_2, errors_2)
regression_2 = fit(log10.(h_2[garde_2]), log10.(errors_2[garde_2]), 1)
println("\nordre de convergence estimé (RK4, système instable) : ", round(regression_2[1], digits = 3))
println("constante K (telle que E ≈ K hᵖ) : K = ", round(10 ^ regression_2[0], digits = 5))

c_2_couleur = palette(:default)[1]
plot(h_2, errors_2; xaxis = :log, yaxis = :log, linecolor = c_2_couleur,
     markershape = :circle, markerstrokecolor = c_2_couleur,
     markercolor = ifelse.(garde_2, c_2_couleur, :white), markerstrokewidth = 1.5,
     xticks = ticks_log10(extrema(h_2)...), yticks = ticks_log10(extrema(errors_2)...),
     xlims = (1e-3, 2e-1), ylims = (1e-4, 1e4),
     xlabel = "Pas h", ylabel = "Erreur globale E", label = "RK4", legend = :topleft)

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Contrôle du pas : un premier essai avec Euler
</div>

Jusqu'ici $h$ était fixe sur tout l'intervalle. Or le bon pas dépend de l'endroit : sur
l'exemple instable ci-dessus, un pas correct près de $t=0$ peut devenir bien trop grand
(ou inutilement petit) plus loin. L'idée d'un **intégrateur à pas variable** est de
choisir, à chaque pas, un $h$ adapté à une **estimation locale de l'erreur**, plutôt que
de fixer $h$ une fois pour toutes.

**Une première estimation d'erreur : le doublement de pas.** Sur $[t_i, t_i+h]$, on
calcule *deux* approximations de $x(t_i+h)$ avec la même méthode (ici Euler, d'ordre $p=1$) :

- $x_1$, obtenue en un seul pas de taille $h$ ;
- $\hat{x}_1$, obtenue en deux pas de taille $h/2$.

Ces deux approximations ont des erreurs locales différentes ($O(h^{p+1})$ pour $x_1$,
plus petite pour $\hat{x}_1$) : leur écart $x_1 - \hat{x}_1$ donne donc une estimation de
l'erreur locale de la moins bonne des deux, que l'on utilisera pour piloter le pas. On garde,
pour poursuivre l'intégration, la valeur la plus précise $\hat{x}_1$ (on parle
d'**extrapolation locale**).

**Tolérance mixte.** Comme $x$ peut avoir des composantes d'amplitudes très différentes, on
compare l'écart à une échelle $\mathrm{sc}_i = \mathrm{abstol} + \mathrm{reltol} \times |x_i|$
propre à chaque composante ($\mathrm{abstol}$ et $\mathrm{reltol}$ sont des tolérances absolue
et relative fixées par l'utilisateur), et on définit l'erreur normalisée

$$
    \mathrm{err} = \sqrt{\frac{1}{n} \sum_{i=1}^n \left(\frac{(x_1)_i - (\hat{x}_1)_i}{\mathrm{sc}_i}\right)^2}.
$$

Le pas est **accepté** si $\mathrm{err} \le 1$, **rejeté** sinon (on recommence alors le même
pas avec un $h$ plus petit, sans avancer en temps).

**Mise à jour du pas.** Dans les deux cas, on ajuste $h$ pour le pas suivant. Puisque
l'erreur locale de la méthode la moins précise se comporte comme $C h^{p+1}$, on a
$\mathrm{err} \approx C h^{p+1} / \mathrm{sc}$, d'où l'idée de choisir le nouveau pas
$h_{\mathrm{new}}$ de façon à viser $\mathrm{err} \approx 1$ :

$$
    h_{\mathrm{new}} = h \times \min\!\Big(\mathrm{fac}_{\max},\ \max\big(\mathrm{fac}_{\min},\
    \sigma\, \mathrm{err}^{-1/(p+1)}\big)\Big),
$$

où $\sigma \in ]0,1[$ (typiquement $0.9$) est un facteur de sécurité (on vise un peu en
dessous de la tolérance, pour limiter le nombre de rejets), et où l'on borne le facteur
d'agrandissement entre $\mathrm{fac}_{\min}$ et $\mathrm{fac}_{\max}$ (typiquement $0.2$ et
$5$) pour éviter des variations trop brutales de $h$ d'un pas à l'autre.

*Référence : Hairer, Nørsett, Wanner (1993), Solving Ordinary Differential Equations I:
Nonstiff Problems, Section II.4.*

### Exercice 10

Compléter la fonction `euler_adapt` ci-dessous, qui implémente cette stratégie de
doublement de pas avec la méthode d'Euler, puis l'exécuter sur l'exemple d'étude pour
différentes tolérances. Combien de pas sont utilisés ? Comparer, pour une précision
comparable, au nombre de pas fixes nécessaires avec la méthode d'Euler classique
(§ La méthode d'Euler).

In [ ]:
"""
    euler_adapt(f, x0, tspan, h0; abstol=1e-6, reltol=1e-3, hmin=1e-10, hmax=Inf)

Intégration adaptative de la méthode d'Euler (ordre 1) par **doublement de
pas** : à chaque itération, on compare un pas `h` et deux pas `h/2`
(extrapolation de Richardson) pour estimer l'erreur locale, sans schéma
emboîté. `h0` est le pas initial. Le pas est accepté (et mis à jour) selon
le contrôle usuel `err = sqrt(mean(((x1 - x1hat) / sc) .^ 2)) ≤ 1`, avec
`sc = abstol + reltol * max(|x|, |x1hat|)` (cf. § Contrôle du pas). Renvoie
les instants `ts` et valeurs `xs` acceptés (vecteurs de longueur variable,
un par pas accepté).
"""
function euler_adapt(f, x0, tspan, h0; abstol = 1e-6, reltol = 1e-3, hmin = 1e-10, hmax = Inf)
    t0, tf = tspan
    t = t0
    x = x0
    h = h0
    ts = [t0]
    xs = [x0]
    p = 1  # ordre de la méthode d'Euler
    while t < tf
        h = min(h, tf - t, hmax)

        x1 = x; x1hat = x # à compléter : un pas h, puis deux pas h/2

        sc = @. abstol + reltol * max(abs(x), abs(x1hat))
        err = 0.0 # à compléter : erreur locale normalisée par sc

        if err ≤ 1 || h ≤ hmin
            t = t + h
            x = x1hat  # extrapolation locale : on garde la valeur la plus précise
            push!(ts, t)
            push!(xs, x)
        end

        # à compléter : mise à jour du pas en fonction de err
        h = max(h, hmin)
    end
    return ts, xs
end

In [ ]:
# Intégration adaptative sur l'exemple d'étude, pour différentes tolérances
for tol in (1e-1, 1e-2, 1e-4, 1e-6)
    ts_a, xs_a = euler_adapt(f, x0, tspan, 0.1; abstol = tol, reltol = tol)
    err_a = maximum(abs.(xs_a .- sol.(ts_a)))
    println("tol = $tol \t pas utilisés = ", length(ts_a) - 1, "\t erreur globale = $err_a")
end

ts_a, xs_a = euler_adapt(f, x0, tspan, 0.1; abstol = 1e-2, reltol = 1e-2)
plt = plot(ts_a, xs_a; marker = :circle, markersize = 2, label = "Euler adaptatif", xlabel = "t", ylabel = "x(t)")
plot!(plt, sol; label = "Solution exacte")

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Une paire de Runge-Kutta emboîtée (RK23 de Bogacki-Shampine)
</div>

Le doublement de pas de l'exercice précédent a un défaut : pour estimer l'erreur, il calcule
**trois fois** le second membre $f$ par pas (une fois pour le pas $h$, deux fois pour les
pas $h/2$), pour un gain de précision finalement modeste (l'ordre reste $1$). Une **paire de
Runge-Kutta emboîtée** évite ce gaspillage : on choisit un tableau de Butcher $(A, c)$ à $s$
étages, mais **deux** jeux de poids $b$ et $\hat{b}$, donnant deux approximations de
$x(t_i+h)$ construites à partir des **mêmes étages** $k_1, \ldots, k_s$ :

$$
    x_1 = x_0 + h \sum_{i=1}^s b_i\, k_i \quad (\text{ordre } p+1), \qquad
    \hat{x}_1 = x_0 + h \sum_{i=1}^s \hat{b}_i\, k_i \quad (\text{ordre } p).
$$

Comme les $k_i$ ne sont calculés qu'une seule fois, on obtient les deux approximations, donc
une estimation de l'erreur locale $x_1 - \hat{x}_1$, **au prix d'un seul jeu de $s$
évaluations de $f$** : c'est strictement le même coût qu'un pas de la méthode à pas fixe
correspondante. Le reste de la stratégie (calcul de $\mathrm{sc}$, de $\mathrm{err}$,
acceptation ou rejet du pas, mise à jour de $h$) est inchangé par rapport à l'exercice
précédent, avec $p$ l'ordre de la méthode la moins précise ($\hat{b}$).

On utilise ici la paire de **Bogacki-Shampine**, d'ordre $3(2)$ (utilisée par exemple par
`ode23` en MATLAB ou `BS3()` en Julia), donnée par le tableau

$$
    \begin{array}{c | c c c c}
        0   & & & & \\
        1/2 & 1/2 & & & \\
        3/4 & 0 & 3/4 & & \\
        1   & 2/9 & 1/3 & 4/9 & \\
        \hline
            & 2/9 & 1/3 & 4/9 & 0 \\
            & 7/24 & 1/4 & 1/3 & 1/8 \\
    \end{array}
$$

où la première ligne sous la barre ($b$) donne la solution propagée, d'ordre $3$, et la
seconde ($\hat{b}$) sert uniquement à l'estimation d'erreur, d'ordre $2$.

### Exercice 11

1. Compléter la fonction `rk23` ci-dessous.
2. L'exécuter sur l'exemple d'étude pour différentes tolérances : comparer le nombre de pas
   utilisés à celui d'`euler_adapt` pour une précision comparable.
3. L'exécuter sur le système instable de l'Exercice 9 (fonctions `f2`, `sol2`). Une tolérance
   fine (par exemple $10^{-9}$) permet-elle de descendre sous l'erreur obtenue en Exercice 9
   avec RK4 à pas fixe, pour un nombre d'évaluations de $f$ comparable ?

In [ ]:
# Tableau de la paire de Bogacki-Shampine (ordre 3(2))
const c23    = [0.0, 1/2, 3/4, 1.0]
const A23    = [0.0 0.0 0.0 0.0;
                1/2 0.0 0.0 0.0;
                0.0 3/4 0.0 0.0;
                2/9 1/3 4/9 0.0]
const b23    = [2/9, 1/3, 4/9, 0.0]     # ordre 3 : solution propagée
const bhat23 = [7/24, 1/4, 1/3, 1/8]    # ordre 2 : estimation d'erreur

"""
    rk23(f, x0, tspan, h0; abstol=1e-6, reltol=1e-3, hmin=1e-10, hmax=Inf)

Intégration adaptative par la **paire de Runge-Kutta emboîtée** de
Bogacki-Shampine, de tableau de Butcher `(A23, b23, c23)` : un seul jeu
d'évaluations de `f` par pas donne à la fois une solution d'ordre 3 (`x1`,
poids `b23`) et une estimation d'ordre 2 (`x1hat`, poids `bhat23`), dont
l'écart sert à estimer l'erreur locale (cf. § Une paire de Runge-Kutta
emboîtée). Le pas accepté et retenu est `x1` (extrapolation locale). Mêmes
conventions d'arguments et de retour que `euler_adapt`.
"""
function rk23(f, x0, tspan, h0; abstol = 1e-6, reltol = 1e-3, hmin = 1e-10, hmax = Inf)
    t0, tf = tspan
    t = t0
    x = x0
    h = h0
    ts = [t0]
    xs = [x0]
    p = 2  # ordre de la méthode la moins précise (bhat23), utilisée pour l'estimation d'erreur
    s = length(c23)
    while t < tf
        h = min(h, tf - t, hmax)

        k = Vector{typeof(x0)}(undef, s)
        for i in 1:s
            xi = x
            for j in 1:i-1
                xi = xi + h * A23[i, j] * k[j]
            end
            k[i] = f(t + c23[i] * h, xi)
        end

        x1 = x; x1hat = x # à compléter : solutions d'ordre 3 et d'ordre 2

        sc = @. abstol + reltol * max(abs(x), abs(x1))
        err = 0.0 # à compléter

        if err ≤ 1 || h ≤ hmin
            t = t + h
            x = x1
            push!(ts, t)
            push!(xs, x)
        end

        # à compléter
        h = max(h, hmin)
    end
    return ts, xs
end

In [ ]:
# Sur l'exemple d'étude
println("Exemple d'étude")
for tol in (1e-1, 1e-2, 1e-4, 1e-6, 1e-9)
    ts23, xs23 = rk23(f, x0, tspan, 0.1; abstol = tol, reltol = tol)
    err23 = maximum(abs.(xs23 .- sol.(ts23)))
    println("tol = $tol \t pas utilisés = ", length(ts23) - 1, "\t erreur globale = $err23")
end

ts_a, xs_a = rk23(f, x0, tspan, 0.1; abstol = 1e-2, reltol = 1e-2)
plt = plot(ts_a, xs_a; marker = :circle, markersize = 2, label = "RK23 adaptatif", xlabel = "t", ylabel = "x(t)")
plot!(plt, sol; label = "Solution exacte")

In [ ]:
# Sur le système instable (Exercice 9)
println("\nSystème instable")
for tol in (1e-1, 1e-2, 1e-4, 1e-6, 1e-9)
    ts23, xs23 = rk23(f2, x0_2, tspan_2, 0.1; abstol = tol, reltol = tol)
    err23 = maximum(norm.(xs23 .- sol2.(ts23)))
    println("tol = $tol \t pas utilisés = ", length(ts23) - 1, "\t erreur globale = $err23")
end

ts_a, xs_a = rk23(f2, x0_2, tspan_2, 0.1; abstol = 1e-10, reltol = 1e-10)
c1, c2 = palette(:default)[1], palette(:default)[2]
plt = plot(ts_a, first.(xs_a); marker = :circle, markersize = 2, color = c1,
           label = "RK23 adaptatif (x₁)", xlabel = "t", ylabel = "x(t)")
plot!(plt, ts_a, last.(xs_a); marker = :circle, markersize = 2, color = c2,
      label = "RK23 adaptatif (x₂)")
plot!(plt, ts_a, first.(sol2.(ts_a)); color = c1, linestyle = :dash, label = "Solution exacte (x₁)")
plot!(plt, ts_a, last.(sol2.(ts_a)); color = c2, linestyle = :dash, label = "Solution exacte (x₂)")

<div style="width:95%;
            margin:10px;
            padding:8px;
            color:white;
            background-color: rgb(46, 109, 4);
            border-radius:10px;
            font-weight:bold;
            font-size:1.5em;
            text-align:center;">
Comparaison avec un solveur professionnel : Tsit5
</div>

`OrdinaryDiffEqTsit5` (déjà utilisé dans le TP « Systèmes dynamiques ») fournit `Tsit5()`,
une paire de Runge-Kutta emboîtée d'ordre $5(4)$ due à Tsitouras, construite exactement sur
le même principe que `rk23` (tableau de Butcher à deux jeux de poids $b$, $\hat{b}$), mais à
$7$ étages, avec en plus la propriété **FSAL** (*first same as last*) : le dernier étage d'un
pas accepté est égal au premier étage du pas suivant, ce qui économise une évaluation de $f$
par pas.

Deux différences de convention avec ce qu'on a écrit jusqu'ici :

- le second membre s'écrit `f(x, p, t)` (état, paramètres, temps), et non `f(t, x)` ;
- on construit un `ODEProblem(f, x0, tspan)` puis on appelle
  `solve(prob, Tsit5(); abstol=..., reltol=...)`, qui renvoie un objet `sol` dont
  `sol.t`, `sol.u` sont les instants et valeurs acceptés, et `sol.stats.nf` le nombre total
  d'évaluations de $f$ effectuées (pas acceptés **et** rejetés).

**Diagramme travail-précision.** Pour comparer équitablement plusieurs intégrateurs, on ne
compare pas leur nombre de pas (un pas de `rk23` coûte $4$ évaluations de $f$, un pas de
`euler_adapt` en coûte $3$, un pas de `Tsit5` entre $6$ et $7$), mais le nombre
**d'évaluations de $f$** en fonction de l'erreur globale obtenue : c'est le *diagramme
travail-précision* (*work-precision diagram*), l'outil standard (Hairer, Nørsett, Wanner)
pour comparer des solveurs à pas variable entre eux.

### Exercice 12

1. Compléter le wrapper `f_ode` ci-dessous, qui adapte `f` à la convention `(x, p, t)`
   attendue par `OrdinaryDiffEqTsit5`.
2. Exécuter la cellule suivante, qui trace le diagramme travail-précision de `euler_adapt`,
   `rk23` et `Tsit5` sur l'exemple d'étude, pour une même gamme de tolérances. Commentaires :
   à budget de calcul égal (même nombre d'évaluations de $f$), quel solveur atteint la
   meilleure précision ?
3. Refaire la comparaison sur le système instable (Exercice 9). `Tsit5` permet-il de suivre
   la solution avec une erreur nettement plus petite que `rk23`, pour un coût comparable ?
4. *Bonus.* En appelant `Tsit5` à pas fixe (`solve(prob, Tsit5(); adaptive=false, dt=h)`),
   vérifier numériquement que son ordre de convergence est bien $5$. Attention : Tsit5 étant
   d'ordre plus élevé que RK4, son erreur atteint le plancher de l'erreur d'arrondi
   (§ Limite de précision) pour un pas $h$ plus grand — un piège classique de la régression
   log-log, déjà signalé en Exercice 8. La cellule ci-dessous réutilise pour cela
   `masque_avant_plancher` (§ Fonctions auxiliaires) : les points en creux, écartés de la
   régression, sont les mêmes que ceux affichés en creux depuis la Comparaison des méthodes.

In [ ]:
# Second membre au format attendu par OrdinaryDiffEqTsit5 : f_ode(x, p, t)
"""
    f_ode(x, p, t)

Second membre au format attendu par `OrdinaryDiffEqTsit5`/SciML,
`ẋ = f_ode(x, p, t)` (paramètre `p` non utilisé ici), ré-écrit à partir de
la convention `f(t, x)` utilisée dans le reste du TP.
"""
function f_ode(x, p, t)
    return 0.0 # à compléter
end

prob = ODEProblem(f_ode, x0, tspan)

In [ ]:
# Diagramme travail-précision sur l'exemple d'étude
tols = 10 .^ range(0, stop = -12, length = 8)

nf_euler = Float64[]; err_euler = Float64[]
nf_rk23  = Float64[]; err_rk23  = Float64[]
nf_tsit5 = Float64[]; err_tsit5 = Float64[]

for tol in tols
    ts_e, xs_e = euler_adapt(f, x0, tspan, 0.1; abstol = tol, reltol = tol)
    push!(nf_euler, 3 * (length(ts_e) - 1))              # 3 évaluations de f par pas
    push!(err_euler, maximum(abs.(xs_e .- sol.(ts_e))))

    ts_r, xs_r = rk23(f, x0, tspan, 0.1; abstol = tol, reltol = tol)
    push!(nf_rk23, 4 * (length(ts_r) - 1))                # 4 évaluations de f par pas
    push!(err_rk23, maximum(abs.(xs_r .- sol.(ts_r))))

    sol_t5 = solve(prob, Tsit5(); abstol = tol, reltol = tol)
    push!(nf_tsit5, sol_t5.stats.nf)
    push!(err_tsit5, maximum(abs.(sol_t5.u .- sol.(sol_t5.t))))
end

plot(nf_euler, err_euler; xaxis = :log, yaxis = :log, marker = :circle, label = "Euler adaptatif")
plot!(nf_rk23, err_rk23; marker = :circle, label = "RK23")
plot!(nf_tsit5, err_tsit5; marker = :circle, label = "Tsit5",
      xticks = ticks_log10(extrema(vcat(nf_euler, nf_rk23, nf_tsit5))...),
      yticks = ticks_log10(extrema(vcat(err_euler, err_rk23, err_tsit5))...),
      ylims = (1e-14, 10),
      xlabel = "Nombre d'évaluations de f", ylabel = "Erreur globale E", legend = :bottomleft,
      title = "Diagramme travail-précision")

In [ ]:
# Même comparaison sur le système instable (Exercice 9)
f2_ode(x, p, t) = f2(t, x)
prob2 = ODEProblem(f2_ode, x0_2, tspan_2)

nf_euler_2 = Float64[]; err_euler_2 = Float64[]
nf_rk23_2  = Float64[]; err_rk23_2  = Float64[]
nf_tsit5_2 = Float64[]; err_tsit5_2 = Float64[]

for tol in tols
    ts_e, xs_e = euler_adapt(f2, x0_2, tspan_2, 0.1; abstol = tol, reltol = tol)
    push!(nf_euler_2, 3 * (length(ts_e) - 1))
    push!(err_euler_2, maximum(norm.(xs_e .- sol2.(ts_e))))

    ts_r, xs_r = rk23(f2, x0_2, tspan_2, 0.1; abstol = tol, reltol = tol)
    push!(nf_rk23_2, 4 * (length(ts_r) - 1))
    push!(err_rk23_2, maximum(norm.(xs_r .- sol2.(ts_r))))

    sol_t5 = solve(prob2, Tsit5(); abstol = tol, reltol = tol)
    push!(nf_tsit5_2, sol_t5.stats.nf)
    push!(err_tsit5_2, maximum(norm.(sol_t5.u .- sol2.(sol_t5.t))))
end

plot(nf_euler_2, err_euler_2; xaxis = :log, yaxis = :log, marker = :circle, label = "Euler adaptatif")
plot!(nf_rk23_2, err_rk23_2; marker = :circle, label = "RK23")
plot!(nf_tsit5_2, err_tsit5_2; marker = :circle, label = "Tsit5",
      xticks = ticks_log10(extrema(vcat(nf_euler_2, nf_rk23_2, nf_tsit5_2))...),
      yticks = ticks_log10(extrema(vcat(err_euler_2, err_rk23_2, err_tsit5_2))...),
      ylims = (1e-4, 1e8),
      xlabel = "Nombre d'évaluations de f", ylabel = "Erreur globale E", legend = :bottomleft,
      title = "Diagramme travail-précision — système instable")

In [ ]:
# Trajectoire de Tsit5 sur le système instable, à comparer à celle de RK23 ci-dessus
sol_t5_2 = solve(prob2, Tsit5(); abstol = 1e-10, reltol = 1e-10)
c1, c2 = palette(:default)[1], palette(:default)[2]
plt = plot(sol_t5_2.t, first.(sol_t5_2.u); marker = :circle, markersize = 2, color = c1,
           label = "Tsit5 (x₁)", xlabel = "t", ylabel = "x(t)")
plot!(plt, sol_t5_2.t, last.(sol_t5_2.u); marker = :circle, markersize = 2, color = c2,
      label = "Tsit5 (x₂)")
plot!(plt, sol_t5_2.t, first.(sol2.(sol_t5_2.t)); color = c1, linestyle = :dash, label = "Solution exacte (x₁)")
plot!(plt, sol_t5_2.t, last.(sol2.(sol_t5_2.t)); color = c2, linestyle = :dash, label = "Solution exacte (x₂)")

In [ ]:
# Bonus : ordre de convergence de Tsit5 à pas fixe
Nspan_t5 = max.(1, round.(Int, (tspan[2] - tspan[1]) ./ hspan_default))
h_actual_t5 = (tspan[2] - tspan[1]) ./ Nspan_t5
errors_t5 = [begin
    sol_t5 = solve(prob, Tsit5(); adaptive = false, dt = h)
    maximum(abs.(sol_t5.u .- sol.(sol_t5.t)))
end for h in h_actual_t5]

garde_t5 = masque_avant_plancher(h_actual_t5, errors_t5)

regression_t5 = fit(log10.(h_actual_t5[garde_t5]), log10.(errors_t5[garde_t5]), 1)
println("ordre de convergence estimé pour Tsit5 (pas fixe) : ", round(regression_t5[1], digits = 3))

c_t5_couleur = palette(:default)[1]
plot(h_actual_t5, errors_t5; xaxis = :log, yaxis = :log, linecolor = c_t5_couleur,
     markershape = :circle, markerstrokecolor = c_t5_couleur,
     markercolor = ifelse.(garde_t5, c_t5_couleur, :white), markerstrokewidth = 1.5,
     xticks = ticks_log10(extrema(h_actual_t5)...), yticks = ticks_log10(extrema(errors_t5)...),
     ylims = (1e-15, 10),
     xlabel = "Pas h", ylabel = "Erreur globale E", label = "Tsit5 (pas fixe)", legend = :topleft)